# 🛒 Voice Cart — Bulletproof Intent Classifier Training

**Why this notebook?**
Colab recently upgraded to Python 3.13, completely breaking older libraries like `tensorflowjs` and older versions of `tensorflow`. This notebook bypasses all of that by silently creating a pristine **Python 3.10** environment using Miniconda in the background to train and convert your model flawlessly.

**Instructions:**
1. Select **Runtime -> Change runtime type -> T4 GPU**.
2. Run the first cell to upload your `dataset.json`.
3. Run the second cell. It will install Miniconda, create the Python 3.10 environment, train the model, convert it, and download `voice_cart_model.zip` directly to your computer!

In [ ]:
from google.colab import files
print("Please upload your dataset.json file...")
uploaded = files.upload()

In [ ]:
%%bash

# 1. Install Miniconda quietly
echo "==> Setting up Python 3.10 Environment... (takes about 1-2 minutes)"
MINICONDA_INSTALLER="Miniconda3-latest-Linux-x86_64.sh"
wget -q https://repo.anaconda.com/miniconda/$MINICONDA_INSTALLER
bash $MINICONDA_INSTALLER -b -f -p /usr/local > /dev/null 2>&1

# 2. Create and activate a Python 3.10 environment
conda create -y -q -n py310 python=3.10 > /dev/null 2>&1
source activate py310

# 3. Install exactly the compatible versions of our libraries
echo "==> Installing dependencies..."
pip install -q sentence-transformers tensorflow==2.15.0 tensorflowjs scikit-learn

# 4. Write the training script to a file
cat << 'EOF' > train_script.py
import json
import os
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflowjs as tfjs

print(f'\n==> Training Model (TensorFlow {tf.__version__})')

with open('dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

texts = [item['text'] for item in dataset]
labels = [item['label'] for item in dataset]

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

print('==> Encoding texts with MiniLM (this might take a minute)...')
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = np.array(model.encode(texts, show_progress_bar=False, batch_size=64))

X_train, X_test, y_train, y_test = train_test_split(
    embeddings, encoded_labels, test_size=0.15, random_state=42, stratify=encoded_labels
)

num_classes = len(label_encoder.classes_)
y_train_onehot = keras.utils.to_categorical(y_train, num_classes)
y_test_onehot = keras.utils.to_categorical(y_test, num_classes)

classifier = keras.Sequential([
    layers.Input(shape=(384,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

classifier.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
classifier.fit(X_train, y_train_onehot, epochs=50, batch_size=32, verbose=0)

loss, acc = classifier.evaluate(X_test, y_test_onehot, verbose=0)
print(f'==> Test Accuracy: {acc:.4f}')

export_dir = 'tfjs_model'
os.makedirs(export_dir, exist_ok=True)
tfjs.converters.save_keras_model(classifier, export_dir)
print(f'==> Model successfully exported to {export_dir}/')
EOF

# 5. Run the training script
python train_script.py

# 6. Zip the results
echo "==> Zipping results..."
zip -r -q voice_cart_model.zip tfjs_model/
echo "==> Done!"


In [ ]:
# 7. Download the zip file to your local computer
from google.colab import files
files.download('voice_cart_model.zip')
print("✅ Download complete! Extract this zip into your public/model/ folder.")